# Phase 2 — Exploratory Data Analysis (EDA)

**Project**: Sarcasm & Irony Detection System using NLP  
**Phase**: 2 — EDA & Dataset Analysis  
**Input**: Cleaned datasets from Phase 1 (`train_clean.csv`, `test_clean.csv`)  

---

This notebook performs a comprehensive EDA of the 4-class sarcasm/irony dataset.  
**No models are trained. No final NLP preprocessing is applied. No external data is used.**

## 1. Imports & Configuration

In [ ]:
import sys
import os
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

# Import EDA modules
from src.eda.analyze_dataset import (
    load_datasets, dataset_overview, dataset_overview_table,
    quality_check, class_distribution, class_balance_analysis
)
from src.eda.text_statistics import (
    compute_text_features, overall_and_classwise_stats,
    text_length_statistics_table, outlier_summary
)
from src.eda.social_media_analysis import (
    extract_social_features, url_analysis, mention_analysis,
    hashtag_analysis, emoji_analysis, punctuation_analysis,
    capitalization_analysis, repeated_char_analysis,
    classwise_comparison_table
)
from src.eda.frequency_analysis import (
    eda_normalize, word_frequency, classwise_word_frequency,
    extract_ngrams, classwise_ngrams, stopword_comparison,
    vocabulary_analysis, tfidf_class_distinctive_terms
)
from src.eda import visualization as viz

print('Imports loaded successfully.')

In [ ]:
# Configuration
TRAIN_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_clean.csv')
TEST_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_clean.csv')
LABEL_MAP_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'label_mapping.json')

CLASS_ORDER = ['Regular', 'Sarcasm', 'Irony', 'Figurative']
CLASS_COLORS = {'Regular': '#4CAF50', 'Sarcasm': '#FF5722', 'Irony': '#2196F3', 'Figurative': '#9C27B0'}

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style='whitegrid', font_scale=1.1)

print(f'Train path: {TRAIN_PATH}')
print(f'Test path:  {TEST_PATH}')

## 2. Load Cleaned Data

In [ ]:
# Load Phase 1 cleaned datasets
train_df, test_df = load_datasets(TRAIN_PATH, TEST_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
print(f'\nLabel mapping:')
with open(LABEL_MAP_PATH) as f:
    label_map = json.load(f)
    print(json.dumps(label_map, indent=2))

## 3. Dataset Overview

In [ ]:
# Dataset overview table
overview = dataset_overview_table(train_df, test_df)
print('Dataset Overview:')
display(overview)

# Detailed overview
train_ov = dataset_overview(train_df, 'Train')
print(f"\nTrain dtypes: {train_ov['dtypes']}")
print(f"Memory: {train_ov['memory_usage_mb']} MB")
print(f"Unique texts: {train_ov['unique_texts']:,}")
print(f"Unique labels: {train_ov['unique_labels']}")

In [ ]:
# Sample records
print('Sample training records:')
display(train_df.head(5))

print('\nSample test records:')
display(test_df.head(5))

## 4. Data Quality Verification

Even though Phase 1 performed validation, we perform a lightweight quality recheck.

In [ ]:
train_quality = quality_check(train_df, 'Train')
test_quality = quality_check(test_df, 'Test')

for qc in [train_quality, test_quality]:
    print(f"\n{qc['dataset']} Set: {qc['quality_status']}")
    print(f"  Missing text: {qc['missing_text']}")
    print(f"  Missing labels: {qc['missing_label']}")
    print(f"  Empty strings: {qc['empty_strings']}")
    print(f"  Duplicate rows: {qc['duplicate_rows']}")
    print(f"  Duplicate texts: {qc['duplicate_texts']}")
    print(f"  Invalid labels: {qc['invalid_labels']}")
    if qc['issues']:
        for issue in qc['issues']:
            print(f"  WARNING: {issue}")
    else:
        print(f"  All checks passed.")

## 5. Class Distribution & Balance Analysis

In [ ]:
# Class distribution
train_dist = class_distribution(train_df, 'Train')
test_dist = class_distribution(test_df, 'Test')

print('Training Set Class Distribution:')
display(train_dist)

print('\nTest Set Class Distribution:')
display(test_dist)

In [ ]:
# Class balance analysis
balance = class_balance_analysis(train_dist, test_dist)

for ds in ['train', 'test']:
    b = balance[ds]
    print(f"\n{ds.upper()} Balance: {b['balance_category']}")
    print(f"  Majority: {b['majority_class']} ({b['majority_count']:,})")
    print(f"  Minority: {b['minority_class']} ({b['minority_count']:,})")
    print(f"  Ratio: {b['majority_minority_ratio']}")
    print(f"  Implication: {b['implication']}")

In [ ]:
# Visualization: Class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, dist, title in [
    (axes[0], train_dist, 'Training Set'),
    (axes[1], test_dist, 'Test Set'),
]:
    colors = [CLASS_COLORS[c] for c in dist['Class'].astype(str)]
    bars = ax.bar(dist['Class'].astype(str), dist['Count'], color=colors, edgecolor='white')
    for bar, pct in zip(bars, dist['Percentage']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{int(bar.get_height()):,}\n({pct:.1f}%)",
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(f'Class Distribution - {title}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(dist['Count']) * 1.15)

plt.tight_layout()
plt.show()

## 6. Text Length Analysis

In [ ]:
# Compute text features
text_features = compute_text_features(train_df)

# Statistics table
stats_table = text_length_statistics_table(text_features)
print('Text Length Statistics:')
display(stats_table)

In [ ]:
# Character count distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(text_features['character_count'], bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(text_features['character_count'].mean(), color='#FF5722', ls='--', lw=2,
                label=f"Mean: {text_features['character_count'].mean():.1f}")
axes[0].set_title('Character Count Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(text_features['word_count'], bins=30, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[1].axvline(text_features['word_count'].mean(), color='#FF5722', ls='--', lw=2,
                label=f"Mean: {text_features['word_count'].mean():.1f}")
axes[1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots by class
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=text_features, x='label', y='character_count',
            order=CLASS_ORDER, palette=CLASS_COLORS, ax=axes[0])
axes[0].set_title('Character Count by Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')

sns.boxplot(data=text_features, x='label', y='word_count',
            order=CLASS_ORDER, palette=CLASS_COLORS, ax=axes[1])
axes[1].set_title('Word Count by Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class')

plt.tight_layout()
plt.show()

In [ ]:
# Outlier analysis
outliers = outlier_summary(text_features)

for col, info in outliers.items():
    print(f"\n{col}:")
    print(f"  IQR bounds: [{info['lower_bound']:.0f}, {info['upper_bound']:.0f}]")
    print(f"  Low outliers: {info['low_outlier_count']}")
    print(f"  High outliers: {info['high_outlier_count']}")
    print(f"  Total: {info['total_outliers']} ({info['outlier_percentage']}%)")
    if info.get('outlier_class_distribution'):
        print(f"  Class distribution: {info['outlier_class_distribution']}")
    print(f"  Note: Outliers NOT removed (sarcasm may involve unusual writing)")

## 7. Social Media Pattern Analysis

In [ ]:
# Extract social media features
social_features = extract_social_features(train_df)

# Summary
url_info = url_analysis(social_features)
mention_info = mention_analysis(social_features)
hashtag_info = hashtag_analysis(social_features)
emoji_info = emoji_analysis(social_features)
punct_info = punctuation_analysis(social_features)
caps_info = capitalization_analysis(social_features)
repeat_info = repeated_char_analysis(social_features)

print(f"URLs: {url_info['tweets_with_urls']:,} tweets ({url_info['percentage_with_urls']}%)")
print(f"Mentions: {mention_info['tweets_with_mentions']:,} tweets ({mention_info['percentage_with_mentions']}%)")
print(f"Hashtags: {hashtag_info['tweets_with_hashtags']:,} tweets ({hashtag_info['percentage_with_hashtags']}%)")
print(f"Emojis: {emoji_info['tweets_with_emojis']:,} tweets ({emoji_info['percentage_with_emojis']}%)")
print(f"Uppercase ratio (avg): {caps_info['avg_uppercase_ratio']}")
print(f"All-caps words: {caps_info['tweets_with_allcaps_words']:,} ({caps_info['pct_with_allcaps_words']}%)")
print(f"Repeated chars: {repeat_info['tweets_with_repeated_chars']:,} ({repeat_info['percentage_with_repeated_chars']}%)")

In [ ]:
# Top hashtags
if hashtag_info.get('top_hashtags'):
    ht_df = pd.DataFrame(hashtag_info['top_hashtags'][:15])
    print('Top 15 Hashtags:')
    display(ht_df)

In [ ]:
# Social media features by class
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

feature_plots = [
    (url_info['by_class'], 'URL Presence (%)', axes[0, 0]),
    (mention_info['by_class'], 'Mention Presence (%)', axes[0, 1]),
    (hashtag_info['by_class'], 'Hashtag Presence (%)', axes[0, 2]),
    (emoji_info['by_class'], 'Emoji Presence (%)', axes[1, 0]),
    (caps_info['allcaps_by_class'], 'All-Caps Words (%)', axes[1, 1]),
]

for data, title, ax in feature_plots:
    classes = [d['class'] for d in data]
    values = [d.get('percentage', d.get('mean', 0)) for d in data]
    colors = [CLASS_COLORS.get(c, '#999') for c in classes]
    ax.bar(classes, values, color=colors, edgecolor='white')
    for i, v in enumerate(values):
        ax.text(i, v + max(values)*0.02, f'{v:.1f}%', ha='center', fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(values) * 1.2 if values else 1)

axes[1, 2].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Word Frequency Analysis

In [ ]:
# EDA-only normalization
analysis_df = train_df.copy()
analysis_df['text_normalized'] = eda_normalize(analysis_df['text'])

# Stopword comparison
sw = stopword_comparison(analysis_df['text_normalized'])
print('Raw top 10 words:')
display(pd.DataFrame(sw['raw_top_words'][:10]))

print('\nTop 10 after stopword removal:')
display(pd.DataFrame(sw['no_stopwords_top_words'][:10]))

In [ ]:
# Top unigrams overall
top_uni = word_frequency(analysis_df['text_normalized'], remove_stopwords=True, top_n=20)

fig, ax = plt.subplots(figsize=(10, 8))
words = [w for w, _ in top_uni]
counts = [c for _, c in top_uni]
ax.barh(range(len(words)), counts, color='#4CAF50', edgecolor='white')
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_title('Top 20 Unigrams (After Stopword Removal)', fontsize=14, fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Class-wise top unigrams
cw_uni = classwise_word_frequency(analysis_df, top_n=10)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, label in zip(axes.flatten(), CLASS_ORDER):
    words = [w for w, _ in cw_uni[label]]
    counts = [c for _, c in cw_uni[label]]
    ax.barh(range(len(words)), counts, color=CLASS_COLORS[label], edgecolor='white')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_title(f'Top Unigrams - {label}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. N-gram Analysis

In [ ]:
# Top bigrams and trigrams
top_bi = extract_ngrams(analysis_df['text_normalized'], n=2, top_n=20)
top_tri = extract_ngrams(analysis_df['text_normalized'], n=3, top_n=20)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, ngrams, title, color in [
    (axes[0], top_bi, 'Top 20 Bigrams', '#2196F3'),
    (axes[1], top_tri, 'Top 20 Trigrams', '#FF5722'),
]:
    words = [ng for ng, _ in ngrams]
    counts = [c for _, c in ngrams]
    ax.barh(range(len(words)), counts, color=color, edgecolor='white')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## 10. Class-Wise Comparison

In [ ]:
# Consolidated comparison table
comparison = classwise_comparison_table(text_features, social_features)
print('Consolidated Class-Wise Comparison:')
display(comparison)

## 11. Vocabulary Analysis

In [ ]:
vocab = vocabulary_analysis(analysis_df, 'text_normalized')

print(f"Total vocabulary: {vocab['total_vocabulary_size']:,} unique words")
print(f"Total word occurrences: {vocab['total_word_occurrences']:,}")
print(f"Rare words (<=2 occurrences): {vocab['rare_words_count']:,} ({vocab['rare_words_percentage']}%)")
print(f"Common vocabulary (all classes): {vocab['common_vocabulary_size']:,}")
print(f"\nClass vocabulary sizes:")
for label, size in vocab['class_vocabulary_sizes'].items():
    unique = vocab['class_specific_words'].get(label, 0)
    print(f"  {label}: {size:,} total, {unique:,} unique to class")

## 12. Optional: Sentiment Exploration

Sentiment analysis is **NOT** a target variable for this project.  
However, the concept of positive words in negative contexts is relevant to sarcasm detection.

**Observation**: Sarcasm often uses positive-looking words ("love", "great", "amazing")  
in negative or contradictory contexts. This makes simple sentiment analysis unreliable  
for sarcasm detection — which is precisely *why* our project exists.

In [ ]:
# Positive words in sarcasm class
positive_words = ['love', 'great', 'wonderful', 'amazing', 'best', 'awesome', 'fantastic', 'brilliant', 'perfect', 'thanks']
sarcasm_texts = analysis_df[analysis_df['label'] == 'Sarcasm']['text_normalized']
regular_texts = analysis_df[analysis_df['label'] == 'Regular']['text_normalized']

results = []
for pw in positive_words:
    s_count = sarcasm_texts.str.contains(pw, na=False).sum()
    r_count = regular_texts.str.contains(pw, na=False).sum()
    results.append({'word': pw, 'sarcasm_count': s_count, 'regular_count': r_count})

pw_df = pd.DataFrame(results)
pw_df['sarcasm_pct'] = (pw_df['sarcasm_count'] / len(sarcasm_texts) * 100).round(2)
pw_df['regular_pct'] = (pw_df['regular_count'] / len(regular_texts) * 100).round(2)
print('Positive words: Sarcasm vs Regular class')
display(pw_df)

## 13. Key Observations

Based on the EDA, the key observations are:

1. **Dataset size**: 81,359 training + 5,527 test records across 4 classes
2. **Class balance**: Training set is approximately balanced; test set is moderately imbalanced
3. **Text length**: Average ~103 characters / ~15 words per tweet
4. **Hashtags**: 100% of tweets contain hashtags (dataset characteristic)
5. **Positive words in sarcasm**: Sarcasm class frequently uses positive-looking words
6. **Social media patterns**: URLs, mentions, punctuation, and capitalization vary by class
7. **Vocabulary**: 71K+ unique words with 68.7% being rare (<=2 occurrences)
8. **Duplicate texts**: ~13K duplicate texts with different labels (annotation ambiguity)

## 14. Preprocessing Recommendations

Based on EDA findings, Phase 3 should test:

| Decision | Recommendation | Rationale |
|----------|---------------|----------|
| Hashtags | PRESERVE initially | Some (#sarcasm, #irony) carry signal |
| Mentions | Replace with @USER | Normalize identity, keep pattern |
| URLs | Remove or replace | URL content not accessible |
| Emojis | Test both | May carry pragmatic meaning |
| Punctuation | Preserve as features | May signal sarcasm |
| Capitalization | Extract ratio, then lowercase | Uppercase ratio may be useful |
| Repeated chars | Normalize + count feature | "soooo" -> "so" but keep count |
| Stopwords | Test both | Context matters in short tweets |
| N-grams | Include bigrams | Distinctive patterns observed |
| Rare words | Apply min_df filter | 68.7% of vocab is rare |

In [ ]:
print('Phase 2 EDA complete.')
print('All reports saved to: data/reports/eda/')
print('All visualizations saved to: results/eda/')
print('\nDo NOT begin Phase 3 without explicit instruction.')